# 01 Extraction

**Project:** Smartwatch Health Analytics — User Risk Segmentation  
**Dataset:** Unclean Smartwatch Health Data (10,000 rows)  
**Goal:** Load the raw dataset, profile its structure, and document quality issues before cleaning.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
RAW_PATH = PROJECT_ROOT / 'data/raw/unclean_smartwatch_health_data.csv'

print('Project root:', PROJECT_ROOT)
print('Raw data path:', RAW_PATH)
print('File exists:', RAW_PATH.exists())

## 1. Load Raw Data

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(10)

## 2. Schema Inspection

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 3. Missing Value Audit

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing.to_string())

## 4. Duplicate Records

In [ ]:
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes}')

## 5. Data Quality Issues Found

In [ ]:
# Activity level — unique values including misspellings
print('Activity Level unique values:')
print(df['Activity Level'].value_counts(dropna=False).to_string())

In [ ]:
# Sleep Duration — flag non-numeric entries like 'ERROR'
sleep_col = 'Sleep Duration (hours)'
non_numeric_sleep = df[pd.to_numeric(df[sleep_col], errors='coerce').isna() & df[sleep_col].notna()]
print(f'Non-numeric sleep duration entries: {len(non_numeric_sleep)}')
print(non_numeric_sleep[sleep_col].value_counts().to_string())

In [ ]:
# Heart Rate — flag physiologically implausible values (< 30 or > 220 BPM)
hr_col = 'Heart Rate (BPM)'
hr_numeric = pd.to_numeric(df[hr_col], errors='coerce')
outliers_hr = df[(hr_numeric < 30) | (hr_numeric > 220)]
print(f'Heart Rate outliers (< 30 or > 220 BPM): {len(outliers_hr)}')
print(outliers_hr[[hr_col]].describe())

In [ ]:
# Blood Oxygen — flag values outside 85–100%
box_col = 'Blood Oxygen Level (%)'
box_numeric = pd.to_numeric(df[box_col], errors='coerce')
outliers_box = df[(box_numeric < 85) | (box_numeric > 100)]
print(f'Blood Oxygen outliers (< 85 or > 100): {len(outliers_box)}')

## 6. Extraction Summary

| Check | Finding |
|---|---|
| Row count | 10,000 |
| Column count | 7 |
| Missing User IDs | Present (rows with blank user_id) |
| Duplicate rows | Checked above |
| Sleep Duration errors | `ERROR` string values present |
| Activity Level misspellings | `Highly_Active`, `Actve`, `Seddentary` found |
| Heart Rate outliers | Values > 220 BPM (physiologically impossible) |

**Next Step:** Proceed to `02_cleaning.ipynb` to fix all issues above.